In [1]:
import json
import re
import string
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

### Load Path

In [5]:
path = Path('../datasets/Entity Recognition in Resumes.json')

Entity_labels = [
    'Name',
    'Designation',
    'Companies worked at', 
    'Location',
    'Email Address', 
    'College Name', 
    'Degree', 
    'Graduation Year',
    'Skills', 
    'Years of Experience'
]

Non_lemma_label ={
    'Name', 
    'Email Address', 
    'Graduation Year', 
    'Location',
    'College Name', 
    'Companies worked at'
}

print(f"Entity types: {len(Entity_labels)}")

Entity types: 10


### Data Parsing

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    """Load all records from a JSONL file."""
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def parse_record(record: dict) -> tuple[str, list[tuple]]:
    """
    Extract (content, spans) from a single record.

    Returns
    -------
    content : str
        Raw resume text.
    spans : list of (start, end, label)
        Character-level offsets where end is EXCLUSIVE (Python slice convention).
        Sorted by start position. UNKNOWN labels are dropped.
    """
    content = record['content']
    spans = []
    for ann in record.get('annotation', []):
        label = ann['label'][0] if ann['label'] else 'UNKNOWN'
        if label == 'UNKNOWN':
            continue
        for pt in ann['points']:
            start = pt['start']
            end   = pt['end'] + 1   # convert inclusive -> exclusive
            # Sanity check: stored text must match the slice
            if content[start:end] != pt['text']:
                continue
            spans.append((start, end, label))

    # Sort by start; resolve any end ties by longest span first
    spans.sort(key=lambda s: (s[0], -s[1]))
    return content, spans


# Load & parse all records
raw_records = load_jsonl(path)
parsed = [parse_record(r) for r in raw_records]

print(f"Records loaded  : {len(parsed)}")
print(f"Total spans     : {sum(len(s) for _, s in parsed)}")
print()

# Inspect first record
content0, spans0 = parsed[0]
print(f"=== Resume 0 (first 200 chars) ===")
print(content0[:200])
print(f"\n=== Spans (first 10) ===")
for start, end, label in spans0[:10]:
    print(f"  [{start:5d}:{end:5d}]  {label:<25}  '{content0[start:end][:60]!s}'")

Records loaded  : 220
Total spans     : 3333

=== Resume 0 (first 200 chars) ===
Abhishek Jha
Application Development Associate - Accenture

Bengaluru, Karnataka - Email me on Indeed: indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a

• To work for an organization which provides me the o

=== Spans (first 10) ===
  [    0:   12]  Name                       'Abhishek Jha'
  [   13:   46]  Designation                'Application Development Associate'
  [   49:   58]  Companies worked at        'Accenture'
  [   60:   69]  Location                   'Bengaluru'
  [   95:  146]  Email Address              'Indeed: indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a
'
  [  372:  405]  Designation                'Application Development Associate'
  [  407:  416]  Companies worked at        'Accenture'
  [  727:  770]  Designation                'B.E in Information science and engineering
'
  [  771:  814]  College Name               'B.v.b college of engineering and technology'
  [  856:  861]  Graduation

### Tokenization & BIO Tagging

World level tokenization (CRF/HMM/BiLSTM)

In [28]:
# Word-level tokenizer

def word_tokenize(text: str) -> list[tuple[str, int, int]]:
    """
    Split text on whitespace, preserving character positions.

    Returns
    -------
    list of (token_str, start, end)  where end is exclusive.
    """
    return [(m.group(), m.start(), m.end())
            for m in re.finditer(r'\S+', text)]


def assign_bio_word(tokens: list[tuple], spans: list[tuple]) -> list[tuple[str, str]]:
    """
    Assign BIO tags to word-level tokens using character-offset spans.

    Strategy
    --------
    For each token, find the first span that fully contains it
    (token_start >= span_start AND token_end <= span_end).
    The tag is B- if no prior token in the same span exists, else I-.
    """
    # Build a fast lookup: character position -> (span_index, label)
    # We iterate spans in order and mark the first token of each span.
    span_first_token: dict[int, bool] = {}   # span_idx -> first-token-seen flag

    tagged = []
    for tok, t_start, t_end in tokens:
        label_tag = 'O'
        for span_idx, (s_start, s_end, label) in enumerate(spans):
            if t_start >= s_start and t_end <= s_end:
                if span_idx not in span_first_token:
                    span_first_token[span_idx] = True
                    label_tag = f'B-{label}'
                else:
                    label_tag = f'I-{label}'
                break
        tagged.append((tok, label_tag))
    return tagged


def word_level_bio(content: str, spans: list[tuple]) -> list[tuple[str, str]]:
    """End-to-end word-level BIO tagging for one resume."""
    tokens = word_tokenize(content)
    return assign_bio_word(tokens, spans)


# Demo: resume 0 
bio_word0 = word_level_bio(content0, spans0)

print(f"{'Token':<40} {'BIO Tag'}")
print("-" * 60)
for tok, tag in bio_word0[:25]:
    print(f"{tok:<40} {tag}")

Token                                    BIO Tag
------------------------------------------------------------
Abhishek                                 B-Name
Jha                                      I-Name
Application                              B-Designation
Development                              I-Designation
Associate                                I-Designation
-                                        O
Accenture                                B-Companies worked at
Bengaluru,                               O
Karnataka                                O
-                                        O
Email                                    O
me                                       O
on                                       O
Indeed:                                  B-Email Address
indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a I-Email Address
•                                        O
To                                       O
work                                     O
for                 

In [29]:
# BIO tag statistics across full dataset
all_bio_word = [word_level_bio(c, s) for c, s in parsed]
tag_counter = Counter(tag for seq in all_bio_word for _, tag in seq)

# Group by label
label_token_counts = defaultdict(lambda: {'B': 0, 'I': 0})
for tag, cnt in tag_counter.items():
    if tag == 'O':
        continue
    prefix, label = tag.split('-', 1)
    label_token_counts[label][prefix] += cnt

summary = pd.DataFrame([
    {'Label': lbl, 
     'B tokens': v['B'], 
     'I tokens': v['I'], 
     'Total tokens': v['B'] + v['I'],
     'Avg tokens/entity': round((v['B'] + v['I']) / v['B'], 2) if v['B'] else 0} for lbl, v in label_token_counts.items()
]).sort_values('Total tokens', ascending=False).reset_index(drop=True)

total_O = tag_counter.get('O', 0)
total_entity = sum(summary['Total tokens'])
total_all = total_O + total_entity

print(f"Total tokens in dataset : {total_all:,}")
print(f"O (outside entity)      : {total_O:,}  ({100*total_O/total_all:.1f}%)")
print(f"Entity tokens           : {total_entity:,}  ({100*total_entity/total_all:.1f}%)")
print()
summary

Total tokens in dataset : 113,925
O (outside entity)      : 102,292  (89.8%)
Entity tokens           : 11,633  (10.2%)



,Label,B tokens,I tokens,Total tokens,Avg tokens/entity
0,Skills,349,5974,6323,18.12
1,Designation,478,793,1271,2.66
2,College Name,286,749,1035,3.62
3,Companies worked at,596,367,963,1.62
4,Degree,257,675,932,3.63
5,Name,223,234,457,2.05
6,Email Address,189,75,264,1.40
7,Graduation Year,225,0,225,1.00
8,Location,82,4,86,1.05
9,Years of Experience,40,37,77,1.93


Sub-Word Tokenization (BERT/DistilBERT)

In [30]:
# Sub-word tokenizer (WordPiece simulation)
# In production replace `wordpiece_tokenize` with:
#   from transformers import AutoTokenizer
#   tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# Common resume vocabulary for the demo splitter
_VOCAB = {
    'software','engineer','engineering','development','application','management','system','data','science','machine','learning','computer','network','database','java','python','java','javascript','design','product','project','team','work','experience','university','college','bachelor','master','information','technology','associate','senior','lead','manager','analyst','developer','architect','director','india','bengal','karnat','mumbai','delhi','hydera','pune','chennai','kolkata','indeed','email','skills','profile','education','company','accenture','infosys','wipro','tata','consultancy','services','limited','private','solutions','global','research','artificial','intelligence','deep','neural','cloud','security','mobile','web','front','back','full','stack','agile','scrum','testing','quality','assurance'
}

def wordpiece_tokenize(word: str, vocab: set = _VOCAB) -> list[str]:
    """
    Greedy left-to-right WordPiece tokenizer (demo approximation).
    Tries to match the longest prefix in vocab, marks remainder with ##.
    In production, replace with HuggingFace tokenizer.
    """
    lower = word.lower()
    if lower in vocab or len(lower) <= 3:
        return [lower]
    # Try longest prefix in vocab
    for split in range(min(len(lower) - 2, 12), 3, -1):
        prefix = lower[:split]
        if prefix in vocab:
            suffix = lower[split:]
            return [prefix, f"##{suffix}"]
    return [lower]     # fall back: keep as single token


def align_bio_subword(word_bio: list[tuple[str, str]]) -> list[tuple[str, str]]:
    """
    Expand word-level BIO tags to sub-word BIO tags.

    Rules
    -----
    - B-X word  → first sub-token gets B-X, rest get I-X
    - I-X word  → all sub-tokens get I-X
    - O   word  → all sub-tokens get O
    """
    result = []
    for word, word_tag in word_bio:
        sub_tokens = wordpiece_tokenize(word)
        for i, sub in enumerate(sub_tokens):
            if word_tag == 'O':
                result.append((sub, 'O'))
            elif word_tag.startswith('B-'):
                label = word_tag[2:]
                result.append((sub, f"B-{label}" if i == 0 else f"I-{label}"))
            else:   # I-X
                result.append((sub, word_tag))
    return result


def subword_bio(content: str, spans: list[tuple]) -> list[tuple[str, str]]:
    """Full sub-word BIO pipeline for one resume."""
    word_bio = word_level_bio(content, spans)
    return align_bio_subword(word_bio)


# Demo
bio_sub0 = subword_bio(content0, spans0)

print(f"Word tokens  : {len(bio_word0)}")
print(f"Sub-word tokens: {len(bio_sub0)}")
print()
print(f"{'Sub-token':<30} {'BIO Tag'}")
print("-" * 50)
for tok, tag in bio_sub0[:30]:
    print(f'{tok:<30} {tag}')

Word tokens  : 227
Sub-word tokens: 237

Sub-token                      BIO Tag
--------------------------------------------------
abhishek                       B-Name
jha                            I-Name
application                    B-Designation
development                    I-Designation
associate                      I-Designation
-                              O
accenture                      B-Companies worked at
bengal                         O
##uru,                         O
karnat                         O
##aka                          O
-                              O
email                          O
me                             O
on                             O
indeed:                        B-Email Address
indeed                         I-Email Address
##.com/r/abhishek-jha/10e7a8cb732bc43a I-Email Address
•                              O
to                             O
work                           O
for                            O
an                         

In [31]:
# Compare word-level vs sub-word on a multi-word entity
sample_pairs = [
    ('Software',     'B-Skills'),
    ('Engineering',  'I-Skills'),
    ('and',          'O'),
    ('Development',  'I-Skills'),
    ('at',           'O'),
    ('Bangalore',    'B-Location'),
]

sub_expanded = align_bio_subword(sample_pairs)

print("=== Word-level BIO ===")
print(f"{'Token':<25} {'Tag'}")
for tok, tag in sample_pairs:
    print(f"{tok:<25} {tag}")

print()
print("=== Sub-word BIO (after WordPiece expansion) ===")
print(f"{'Sub-token':<25} {'Tag'}")
for tok, tag in sub_expanded:
    print(f"{tok:<25} {tag}")

=== Word-level BIO ===
Token                     Tag
Software                  B-Skills
Engineering               I-Skills
and                       O
Development               I-Skills
at                        O
Bangalore                 B-Location

=== Sub-word BIO (after WordPiece expansion) ===
Sub-token                 Tag
software                  B-Skills
engineering               I-Skills
and                       O
development               I-Skills
at                        O
bangalore                 B-Location
